<a href="https://colab.research.google.com/github/Ratludu/Backpack-Prediction-Challenge/blob/main/Backpack_Prices_LB_38.84049.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [2]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggleusername')
os.environ['KAGGLE_KEY'] = userdata.get('kaggleapi')

competition = 'playground-series-s5e2'

!kaggle competitions download -c {competition}

!unzip "{competition}.zip"

 99% 92.0M/92.7M [00:05<00:00, 24.1MB/s]
100% 92.7M/92.7M [00:05<00:00, 18.4MB/s]
Archive:  playground-series-s5e2.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
  inflating: training_extra.csv      


In [3]:
import wandb

wandb.login(key = userdata.get("WB"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ratludu (ratludu-backpack-S5E2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import kagglehub

# Download latest version
path1 = kagglehub.dataset_download("souradippal/student-bag-price-prediction-dataset")

print("Path to dataset files:", path1)

100%|██████████| 1.23M/1.23M [00:00<00:00, 2.51MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/souradippal/student-bag-price-prediction-dataset/versions/1


In [5]:
!pip install dask-cuda==24.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.4/134.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.5/244.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2024.10.0
    Uninstalling dask-2024.10.0:
      Successfully uninstalled dask-2024.10.0


In [6]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 582, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 582 (delta 119), reused 82 (delta 82), pack-reused 434 (from 3)
Receiving objects: 100% (582/582), 190.86 KiB | 21.21 MiB/s, done.
Resolving deltas: 100% (293/293), done.
Installing RAPIDS remaining 24.12.* libraries
Using Python 3.11.11 environment at: /usr
Resolved 154 packages in 8.88s
 Downloaded libucx-cu12
 Downloaded datashader
 Downloaded ucx-py-cu12
 Downloaded cuspatial-cu12
 Downloaded cucim-cu12
 Downloaded scikit-image
 Downloaded libcuspatial-cu12
 Downloaded raft-dask-cu12
 Downloaded cuml-cu12
 Downloaded cuvs-cu12
 Downloaded cugraph-cu12
Prepared 21 packages in 20.52s
Uninstalled 1 package in 26ms
Installed 21 packages in 18ms
 + cucim-cu12==24.12.0
 + cugraph-cu12==24.12.0
 + cuml-cu12==24.12.0
 + cuproj-cu12==24.12.0
 + cuspatial-cu12==24.12.0
 + cuvs-cu12==24.12.0
 + cuxfilter-cu1

In [7]:
!pip install catboost
!pip install optuna
!pip install scikit-learn
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.9 MB/s eta 0:00:00


In [8]:
import pandas as pd
import numpy as np
from numpy import random
from cuml.preprocessing import TargetEncoder
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [9]:
# Initialise Project

wandb.init(
    # set the wandb entity where your project will be logged (generally your team name)
    entity="ratludu-backpack-S5E2",

    # set the wandb project where this run will be logged
    project="Catboost-Backpack",

    # track hyperparameters and run metadata
    config={
            #'learning_rate': 0.02,
            #'l2_leaf_reg':5,
            'per_float_feature_quantization':'8:border_count=1024',
            'iterations': 2_000,
            'task_type': "GPU",
            'grow_policy': 'Lossguide',
            'random_state': 42,
            'early_stopping_rounds':250,
            'verbose': 250,
            'loss_function':'RMSE'
    }
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


In [10]:
class config:
    # data links
    train_link = "train.csv"
    train_ex_link = "training_extra.csv"
    test_link = "test.csv"
    sub_link = "sample_submission.csv"

    # create folds config

    n_splits = 25

    # Ignore Columns

    col_ignore = ["id", "Price"]
    num_cols = ["Weight Capacity (kg)"]
    # target

    submit = True

    target = "Price"

    add_original = False

In [11]:
def rmse(y_true, y_pred):
    error = 0

    for yt, yp in zip(y_true, y_pred):
        error += (yt - yp) ** 2

    m = np.sqrt(error / len(y_true))

    return m

In [12]:
def random_columns(columns):

  # Generate random number for how many columns we want to concat
  rand_num = np.random.randint(2,5)

  # choose the columns from the list of columns with no repeats
  rand_cols = []
  for i in range(rand_num):
    col = np.random.choice(columns)
    while col in rand_cols:
      col = np.random.choice(columns)
    rand_cols.append(col)

  # return a list of the columns

  return "-".join(col for col in rand_cols),rand_cols


In [13]:
train = pd.read_csv(config.train_link)
train_ex = pd.read_csv(config.train_ex_link)
test = pd.read_csv(config.test_link)
original = pd.read_csv(path1+"/Noisy_Student_Bag_Price_Prediction_Dataset.csv")
original.dropna(subset=['Price'],inplace = True)

In [14]:
train = pd.concat([train,train_ex, original], axis = 0, ignore_index = True)

In [15]:
#train = train.sample(frac = 0.1, random_state = 42, ignore_index = True)

In [16]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment','Waterproof', 'Style', 'Color','Weight Capacity (kg)']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
cat3_oof = np.zeros(len(train))
cat3_preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features+['Price']].reset_index(drop=True).copy(), train.loc[test_idx, features].reset_index(drop=True).copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    x_train["weight_log"] = np.log1p(x_train["Weight Capacity (kg)"])**2
    x_val["weight_log"] = np.log1p(x_val["Weight Capacity (kg)"])**2
    x_test["weight_log"] = np.log1p(x_test["Weight Capacity (kg)"])**2

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    x_train['Material_TE-Color_TE'] = x_train['Material_TE']*x_train['Color_TE']
    x_val['Material_TE-Color_TE'] = x_val['Material_TE']*x_val['Color_TE']
    x_test['Material_TE-Color_TE'] = x_test['Material_TE']*x_test['Color_TE']

    x_train['Sizexbrand-Materialxweight'] = (x_train['Sizexbrand']-x_train['Materialxweight'])**2
    x_val['Sizexbrand-Materialxweight'] = (x_val['Sizexbrand']-x_val['Materialxweight'])**2
    x_test['Sizexbrand-Materialxweight'] = (x_test['Sizexbrand']-x_test['Materialxweight'])**2

    x_train['Sizexweight-Laptop Compartment_TE'] = x_train['Sizexweight']/x_train['Laptop Compartment_TE']
    x_val['Sizexweight-Laptop Compartment_TE'] = x_val['Sizexweight']/x_val['Laptop Compartment_TE']
    x_test['Sizexweight-Laptop Compartment_TE'] = x_test['Sizexweight']/x_test['Laptop Compartment_TE']

    x_train['Color_TE-Waterproof_TE'] = x_train['Color_TE']**2+x_train['Waterproof_TE']*x_train['Color_TE']
    x_val['Color_TE-Waterproof_TE'] = x_val['Color_TE']**2+x_val['Waterproof_TE']*x_val['Color_TE']
    x_test['Color_TE-Waterproof_TE'] = x_test['Color_TE']**2+x_test['Waterproof_TE']*x_test['Color_TE']

    features2 = list(x_val.columns)
    kf2 = KFold(n_splits=5, shuffle=True, random_state=42)
    for j, (train_index2, test_index2) in enumerate(kf2.split(x_train)):
        print(f" ## INNER Fold {j+1} (outer fold {fold+1}) ##")

        x_train2 = x_train.loc[train_index2,features2+['Price']].copy()
        x_valid2 = x_train.loc[test_index2,features2].copy()
        for col in drop:
          aggs = {}

          aggs['Price'] = ["std","median","min","max","skew"]

          train_agg = x_train2.groupby([col]).agg(aggs)
          train_agg = train_agg.droplevel(axis=1, level =0).reset_index()
          col_names = {s:col+"_"+s+"_aggs_price" for s in ["std","median","min","max","skew"]}
          train_agg.rename(columns = col_names, inplace = True)

          x_valid2 = x_valid2.merge(train_agg, how='left', on = col)

          for c in list(col_names.values()):
            x_train.loc[test_index2,c] = x_valid2[c].values

    for col in drop:
      aggs = {}

      aggs['Price'] = ["std","median","min","max","skew"]

      train_agg = x_train.groupby([col]).agg(aggs)
      train_agg = train_agg.droplevel(axis=1, level =0).reset_index()
      col_names = {s:col+"_"+s+"_aggs_price" for s in ["std","median","min","max","skew"]}
      train_agg.rename(columns = col_names, inplace = True)

      x_val = x_val.merge(train_agg, how='left', on = col)
      x_test = x_test.merge(train_agg, how='left', on = col)

    for col in drop:

      if col == 'Weight Capacity (kg)':
        pass

      else:
        aggs = {}

        aggs['Weight Capacity (kg)'] = ["std","median","min","max","skew"]

        train_agg = x_train.groupby([col]).agg(aggs)
        train_agg = train_agg.droplevel(axis=1, level =0).reset_index()
        col_names = {s:col+"_"+s+"_aggs_weight" for s in ["std","median","min","max","skew"]}
        train_agg.rename(columns = col_names, inplace = True)

        x_train = x_train.merge(train_agg, how='left', on = col)
        x_val = x_val.merge(train_agg, how='left', on = col)
        x_test = x_test.merge(train_agg, how='left', on = col)


    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('str')
        x_val[cat] = x_val[cat].astype('str')
        x_test[cat] = x_test[cat].astype('str')

    x_train.drop(['Price'], axis = 1, inplace = True)

    print(x_train.columns)

    model = CatBoostRegressor(**wandb.config, cat_features = cats)

    model.fit(x_train, y_train, eval_set=(x_val,y_val))

    val_preds = model.predict(x_val)

    cat3_oof[test_idx] = val_preds

    cat3_preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')
    wandb.log({"RMSE": score})

print(f"The average CV is {np.average(m)}")
wandb.log({"Average_Score":np.average(m)})
wandb.finish()

 ## INNER Fold 1 (outer fold 1) ##
 ## INNER Fold 2 (outer fold 1) ##
 ## INNER Fold 3 (outer fold 1) ##
 ## INNER Fold 4 (outer fold 1) ##
 ## INNER Fold 5 (outer fold 1) ##
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe',
       ...
       'Style_std_aggs_weight', 'Style_median_aggs_weight',
       'Style_min_aggs_weight', 'Style_max_aggs_weight',
       'Style_skew_aggs_weight', 'Color_std_aggs_weight',
       'Color_median_aggs_weight', 'Color_min_aggs_weight',
       'Color_max_aggs_weight', 'Color_skew_aggs_weight'],
      dtype='object', length=118)
Learning rate set to 0.124435
0:	learn: 38.8829276	test: 38.8819832	best: 38.8819832 (0)	total: 54.8ms	remaining: 1m 49s
250:	learn: 38.6016517	test: 38.6440122	best: 38.6439738 (233)	total: 9.61s	remaining: 1m 6s
500:	learn: 38.5614666	test: 38.6430037	best: 38.6427286 (475)	total: 18s	remaining: 53.9s
750:	le

Average_Score,▁
RMSE,▄▃▃▂▇▂▁▅▄▄▆▆▃▇█▄▂▁▁▃▃▆▅▅█
Average_Score,38.64448
RMSE,38.74108


In [17]:
submission = pd.read_csv(config.sub_link)
submission[config.target] = cat3_preds
submission.to_csv("submission.csv", index = False)

submission

,id,Price
0,300000,81.411495
1,300001,82.716422
2,300002,87.158659
3,300003,79.640173
4,300004,79.447069
...,...,...
199995,499995,81.408388
199996,499996,78.618412
199997,499997,82.740578
199998,499998,82.187661


In [18]:
if config.submit:
  !kaggle competitions submit -c {competition} -f submission.csv -m 'Submission'

100% 4.74M/4.74M [00:02<00:00, 2.11MB/s]
Successfully submitted to Backpack Prediction Challenge

In [19]:
!kaggle competitions submissions -c {competition}

fileName           date                 description                                                   status    publicScore  privateScore  
-----------------  -------------------  ------------------------------------------------------------  --------  -----------  ------------  
submission.csv     2025-02-26 13:07:09  Submission                                                    pending                              
submission.csv     2025-02-25 13:50:03  Submission                                                    complete  38.84492                   
submission.csv     2025-02-25 11:51:23  Submission with cv 38.6498317592693                           complete  38.84466                   
submission.csv     2025-02-25 10:16:44  Submission with cv 38.6498317592693                           complete  38.84544                   
submission.csv     2025-02-25 02:10:31  Submission with cv 38.6498317592693                           complete  38.84519                   
submission.csv     2

In [20]:
from google.colab import runtime
runtime.unassign()